In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
%%writefile train.py
"""QLoRA fine-tuning of a small code model on Spider.

Usage: python train.py            (env vars: MAX_TRAIN=3000, EPOCHS=1, LORA_R=16, USE_WANDB=1)
Output: adapter saved to outputs/adapter
"""
import json
import os

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

MODEL = os.environ.get("MODEL", "Qwen/Qwen2.5-Coder-1.5B-Instruct")
MAX_TRAIN = int(os.environ.get("MAX_TRAIN", 3000))
EPOCHS = float(os.environ.get("EPOCHS", 1))
LORA_R = int(os.environ.get("LORA_R", 16))
MAX_LEN = 768
OUT_DIR = "outputs/adapter"

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

rows = [json.loads(line) for line in open("data/train.jsonl")][:MAX_TRAIN]


def encode(r):
    p = tok(r["prompt"], add_special_tokens=False)["input_ids"]
    a = tok(r["query"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p + a)[:MAX_LEN]
    labels = ([-100] * len(p) + a)[:MAX_LEN]  # loss only on the SQL answer
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}


ds = Dataset.from_list(rows).map(encode, remove_columns=list(rows[0].keys()))
ds = ds.filter(lambda x: any(l != -100 for l in x["labels"]))
print("training examples:", len(ds))

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={"": 0})
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(
        r=LORA_R,
        lora_alpha=2 * LORA_R,
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ),
)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="outputs/checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,  # T4 GPUs on Kaggle/Colab do not support bf16
    logging_steps=10,
    save_strategy="no",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="wandb" if os.environ.get("USE_WANDB") else "none",
    run_name=f"text2sql-r{LORA_R}",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
)
trainer.train()
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("saved adapter to", OUT_DIR)


Overwriting train.py


In [23]:
%%writefile eval.py

"""Execution-accuracy evaluation on Spider dev, with an optional error-feedback retry.

Usage:
  python eval.py /path/to/spider baseline                       # base model, no adapter
  python eval.py /path/to/spider finetuned outputs/adapter      # with a LoRA adapter

Env vars:
  N_EVAL=200   number of dev questions (0 = all 1034)
  SHUFFLE=0    1 = shuffle dev with a fixed seed (42) before taking N_EVAL.
               Use the SAME setting for baseline and fine-tuned runs, or the comparison is unfair.
  RETRY=1      1 = if the first SQL fails to execute, send the error back to the model once
  MODEL=Qwen/Qwen2.5-Coder-1.5B-Instruct

Metric: a prediction is correct if it returns the same rows as the gold SQL on the real
SQLite database (row order ignored). This is a simplified version of Spider's official
test-suite evaluation, so call it "execution accuracy (simplified)".

Reports two numbers: first-attempt accuracy, and accuracy after the retry.
"""
import json
import os
import random
import sqlite3
import sys
import time
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SPIDER = Path(sys.argv[1])
TAG = sys.argv[2]
ADAPTER = sys.argv[3] if len(sys.argv) > 3 else None
MODEL = os.environ.get("MODEL", "Qwen/Qwen2.5-Coder-1.5B-Instruct")
N_EVAL = int(os.environ.get("N_EVAL", 200))
SHUFFLE = int(os.environ.get("SHUFFLE", 0))
RETRY = int(os.environ.get("RETRY", 1))
BATCH = 8

rows = [json.loads(line) for line in open("data/dev.jsonl")]
if SHUFFLE:
    random.Random(42).shuffle(rows)
if N_EVAL > 0:
    rows = rows[:N_EVAL]

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={"": 0})
if ADAPTER:
    from peft import PeftModel

    model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()


def clean(text):
    text = text.split("###")[0].strip()
    text = text.replace("```sql", "").replace("```", "").strip()
    return " ".join(text.split())


def generate(prompts):
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=160, do_sample=False, pad_token_id=tok.pad_token_id)
    texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [clean(t) for t in texts]


def retry_prompt(prompt, err):
    """Same prompt, with the failed attempt's error inserted before the final '### SQL:' marker."""
    base = prompt[: -len("### SQL:\n")]
    return base + f"### Previous attempt failed with: {err}\n\n### SQL:\n"


def execute(db_id, sql, timeout=10):
    path = SPIDER / "database" / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(f"file:{path}?mode=ro", uri=True)
    conn.text_factory = lambda b: b.decode(errors="ignore")
    deadline = time.time() + timeout
    conn.set_progress_handler(lambda: 1 if time.time() > deadline else 0, 10000)
    try:
        return Counter(map(repr, conn.execute(sql).fetchall())), None
    except Exception as e:  # noqa: BLE001
        return None, str(e)
    finally:
        conn.close()


def is_correct(item):
    return item["gold"] is not None and item["got"] is not None and item["gold"] == item["got"]


first_ok = final_ok = n_retried = n_fixed = 0
results = []
for i in range(0, len(rows), BATCH):
    batch = rows[i : i + BATCH]
    preds = generate([r["prompt"] for r in batch])

    items = []
    for r, pred in zip(batch, preds):
        gold, _ = execute(r["db_id"], r["query"])
        got, err = execute(r["db_id"], pred)
        item = {"r": r, "gold": gold, "pred": pred, "got": got, "err": err,
                "first_pred": pred, "first_err": err, "retried": False}
        item["first_correct"] = is_correct(item)
        items.append(item)

    if RETRY:
        bad = [it for it in items if it["err"]]
        if bad:
            new_preds = generate([retry_prompt(it["r"]["prompt"], it["err"]) for it in bad])
            for it, pred in zip(bad, new_preds):
                got, err = execute(it["r"]["db_id"], pred)
                it.update(pred=pred, got=got, err=err, retried=True)

    for it in items:
        correct = is_correct(it)
        first_ok += it["first_correct"]
        final_ok += correct
        n_retried += it["retried"]
        n_fixed += it["retried"] and correct
        results.append(
            {"db_id": it["r"]["db_id"], "question": it["r"]["question"], "gold": it["r"]["query"],
             "first_pred": it["first_pred"], "first_error": it["first_err"],
             "first_correct": it["first_correct"],
             "pred": it["pred"], "error": it["err"], "correct": correct, "retried": it["retried"]}
        )
    done = len(results)
    print(f"{done}/{len(rows)}  first-attempt acc: {first_ok / done:.3f}  with retry: {final_ok / done:.3f}")

n = len(results)
print(f"\n[{TAG}] n={n} shuffle={SHUFFLE}")
print(f"  first-attempt execution accuracy: {first_ok / n:.3f} ({first_ok}/{n})")
if RETRY:
    print(f"  with error-feedback retry:        {final_ok / n:.3f} ({final_ok}/{n})  "
          f"[retried {n_retried}, fixed {n_fixed}]")
Path("outputs").mkdir(exist_ok=True)
json.dump(
    {"tag": TAG, "n": n, "shuffle": SHUFFLE, "retry": RETRY,
     "accuracy_first_attempt": first_ok / n, "accuracy_with_retry": final_ok / n,
     "retried": n_retried, "fixed_by_retry": n_fixed, "results": results},
    open(f"outputs/results_{TAG}.json", "w"), indent=2)

Overwriting eval.py


In [6]:
%%writefile prepare_data.py
"""Build prompt files from the Spider dataset.

Usage: python prepare_data.py /path/to/spider
Expects: train_spider.json, dev.json, tables.json, database/ inside that folder.
Writes: data/train.jsonl, data/dev.jsonl
"""
import json
import sys
from pathlib import Path

SPIDER = Path(sys.argv[1] if len(sys.argv) > 1 else "spider")
OUT = Path("data")
OUT.mkdir(exist_ok=True)


def load_schemas():
    schemas = {}
    for db in json.load(open(SPIDER / "tables.json")):
        cols = db["column_names_original"]
        types = db["column_types"]
        tables = db["table_names_original"]
        per_table = {i: [] for i in range(len(tables))}
        for idx, (t, c) in enumerate(cols):
            if t >= 0:
                per_table[t].append(f"{c} {types[idx]}")
        lines = [f"CREATE TABLE {tables[i]} ({', '.join(per_table[i])});" for i in per_table]
        for a, b in db["foreign_keys"]:
            lines.append(
                f"-- FK: {tables[cols[a][0]]}.{cols[a][1]} -> {tables[cols[b][0]]}.{cols[b][1]}"
            )
        schemas[db["db_id"]] = "\n".join(lines)
    return schemas


def build_prompt(schema, question):
    return f"### Schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"


def convert(src, dst, schemas):
    data = json.load(open(SPIDER / src))
    with open(OUT / dst, "w") as f:
        for ex in data:
            row = {
                "db_id": ex["db_id"],
                "question": ex["question"],
                "query": ex["query"],
                "prompt": build_prompt(schemas[ex["db_id"]], ex["question"]),
            }
            f.write(json.dumps(row) + "\n")
    print(f"{dst}: {len(data)} examples")


if __name__ == "__main__":
    schemas = load_schemas()
    convert("train_spider.json", "train.jsonl", schemas)
    convert("dev.json", "dev.jsonl", schemas)


Overwriting prepare_data.py


In [7]:
%%writefile serve.py
"""FastAPI service: natural-language question -> validated, read-only SQL result.

Run: DB_PATH=my.sqlite ADAPTER=outputs/adapter uvicorn serve:app --port 8000
"""
import os
import sqlite3

import sqlglot
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from sqlglot import exp
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = os.environ.get("MODEL", "Qwen/Qwen2.5-Coder-1.5B-Instruct")
ADAPTER = os.environ.get("ADAPTER", "outputs/adapter")
DB_PATH = os.environ["DB_PATH"]

tok = AutoTokenizer.from_pretrained(MODEL)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
from peft import PeftModel  # noqa: E402

model = PeftModel.from_pretrained(model, ADAPTER).eval()
app = FastAPI(title="Text-to-SQL")


def get_schema():
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    rows = conn.execute("SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL").fetchall()
    conn.close()
    return "\n".join(r[0] + ";" for r in rows)


def validate(sql):
    """Allow a single SELECT statement only."""
    try:
        parsed = sqlglot.parse(sql, read="sqlite")
    except sqlglot.errors.ParseError as e:
        raise ValueError(f"unparseable SQL: {e}")
    if len(parsed) != 1 or not isinstance(parsed[0], (exp.Select, exp.Union)):
        raise ValueError("only a single SELECT statement is allowed")
    forbidden = (exp.Insert, exp.Update, exp.Delete, exp.Drop, exp.Create, exp.Command)
    if any(parsed[0].find(t) for t in forbidden):
        raise ValueError("write operations are not allowed")


def generate_sql(question, error_hint=None):
    prompt = f"### Schema:\n{get_schema()}\n\n### Question:\n{question}\n\n"
    if error_hint:
        prompt += f"### Previous attempt failed with: {error_hint}\n\n"
    prompt += "### SQL:\n"
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=160, do_sample=False)
    text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return " ".join(text.split("###")[0].replace("```sql", "").replace("```", "").split())


def run_readonly(sql):
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, timeout=5)
    try:
        cur = conn.execute(sql)
        return [d[0] for d in cur.description], cur.fetchmany(100)
    finally:
        conn.close()


class Ask(BaseModel):
    question: str


@app.post("/ask")
def ask(body: Ask):
    sql, error = generate_sql(body.question), None
    for attempt in range(2):  # one retry, feeding the error back to the model
        try:
            validate(sql)
            cols, rows = run_readonly(sql)
            return {"sql": sql, "columns": cols, "rows": rows, "retried": attempt == 1}
        except Exception as e:  # noqa: BLE001
            error = str(e)
            if attempt == 0:
                sql = generate_sql(body.question, error_hint=error)
    raise HTTPException(status_code=422, detail={"sql": sql, "error": error})


Writing serve.py


In [8]:
%cd /kaggle/working
!ls
!find /kaggle/input -name "dev.json"

/kaggle/working
eval.py  prepare_data.py  serve.py  train.py


In [11]:
!find /kaggle/input -name "dev.json"
!find /kaggle/input -name "tables.json"
!find /kaggle/input -name "train_spider.json"

/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/dev.json
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/tables.json
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/train_spider.json


In [10]:
!nvidia-smi
!pip install -q -U bitsandbytes peft accelerate datasets sqlglot

Wed Oct 14 07:59:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
!ls /kaggle/input
!find /kaggle/input -name "dev.json"

datasets  notebooks
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/dev.json


In [13]:
!ls /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider

database      dev.json	  tables.json	  train_others.json
dev_gold.sql  README.txt  train_gold.sql  train_spider.json


In [14]:
%cd /kaggle/working
!python prepare_data.py /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider
!head -c 1500 data/train.jsonl

/kaggle/working
train.jsonl: 7000 examples
dev.jsonl: 1034 examples
{"db_id": "department_management", "question": "How many heads of the departments are older than 56 ?", "query": "SELECT count(*) FROM head WHERE age  >  56", "prompt": "### Schema:\nCREATE TABLE department (Department_ID number, Name text, Creation text, Ranking number, Budget_in_Billions number, Num_Employees number);\nCREATE TABLE head (head_ID number, name text, born_state text, age number);\nCREATE TABLE management (department_ID number, head_ID number, temporary_acting text);\n-- FK: management.head_ID -> head.head_ID\n-- FK: management.department_ID -> department.Department_ID\n\n### Question:\nHow many heads of the departments are older than 56 ?\n\n### SQL:\n"}
{"db_id": "department_management", "question": "List the name, born state and age of the heads of departments ordered by age.", "query": "SELECT name ,  born_state ,  age FROM head ORDER BY age", "prompt": "### Schema:\nCREATE TABLE department (Departme

In [15]:
!N_EVAL=16 python eval.py /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider smoketest

config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 4.13MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 27.5MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 51.3MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 119MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 145MB/s]

model.safetensors: downloading bytes: ▌                     | 79.7MB, 4.86MB/s  
model.safetensors: downloading bytes: ███▉                  |  559MB, 46.0MB/s  
model.safetensors: downloading bytes: █████                 |  712MB, 59.2MB/s  
model.safetensors: reconstructing file:  18%|█     |  559MB / 3.09GB, 24.6MB/s  
model.safetensors: downloading bytes: ███████               |  989MB, 80.9MB/s  
model.safetensors: downloading bytes: ███████▊              | 1.10GB, 89.3MB/s  
model.safetensors: downloading bytes: █████████▍            | 1.33GB,  107MB/s  
model.safetensors: download

In [16]:
import json
r = json.load(open("outputs/results_smoketest.json"))["results"]
for x in r[:3]:
    print(x["question"]); print("GOLD:", x["gold"]); print("PRED:", x["pred"]); print()

How many singers do we have?
GOLD: SELECT count(*) FROM singer
PRED: SELECT count(*) FROM singer;

What is the total number of singers?
GOLD: SELECT count(*) FROM singer
PRED: SELECT count(*) FROM singer;

Show name, country, age for all singers ordered by age from the oldest to the youngest.
GOLD: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
PRED: SELECT name , country , age FROM singer ORDER BY age ASC;



In [17]:
!N_EVAL=200 python eval.py /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider baseline

Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 318.22it/s]
8/200  acc so far: 0.750
16/200  acc so far: 0.688
24/200  acc so far: 0.667
32/200  acc so far: 0.750
40/200  acc so far: 0.675
48/200  acc so far: 0.667
56/200  acc so far: 0.607
64/200  acc so far: 0.609
72/200  acc so far: 0.556
80/200  acc so far: 0.537
88/200  acc so far: 0.557
96/200  acc so far: 0.562
104/200  acc so far: 0.519
112/200  acc so far: 0.491
120/200  acc so far: 0.475
128/200  acc so far: 0.461
136/200  acc so far: 0.449
144/200  acc so far: 0.444
152/200  acc so far: 0.441
160/200  acc so far: 0.425
168/200  acc so far: 0.417
176/200  acc so far: 0.409
184/200  acc so far: 0.397
192/200  acc so far: 0.417
200/200  acc so far: 0.425

[baseline] execution accuracy: 0.425 (85/200)


In [20]:
!MAX_TRAIN=3000 EPOCHS=1 python train.py

Filter: 100%|█████████████████████| 3000/3000 [00:00<00:00, 21582.97 examples/s]
training examples: 3000
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 301.98it/s]
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
{'loss': '1', 'grad_norm': '1.815', 'learning_rate': '0.00018', 'epoch': '0.1067'}
{'loss': '0.4387', 'grad_norm': '0.6468', 'learning_rate': '0.0001944', 'epoch': '0.2133'}
{'loss': '0.4073', 'grad_norm': '0.7087', 'learning_rate': '0.0001758', 'epoch': '0.32'}
{'loss': '0.3643', 'grad_norm': '0.6884', 'learning_rate': '0.0001467', 'epoch': '0.4267'}
{'loss': '0.3183', 'grad_norm': '0.7312', 'learning_rate': '0.0001112', 'epoch': '0.5333'}
{'loss': '0.3023', 'grad_norm': '0.6476', 'learning_rate': '7.412e-05', 'epoch': '0.64'}
{'loss': '0.3096', 'grad_norm': '0.4301', 'learning_rate': '4.062e-05', 'epoch': '0.7467'}
{'loss': '0.2617', 'grad_norm': '0.5653', 'learning_rate': '1.533e-05', 'epoch': '0.8533'}
{'loss': '0.2992',

In [19]:
!sed -i 's/warmup_ratio=0.03/warmup_steps=10/' train.py
!grep -n "warmup" train.py

74:    warmup_steps=10,


In [21]:
!ls -lh outputs/adapter

total 82M
-rw-r--r-- 1 root root 1.2K Oct 14 08:46 adapter_config.json
-rw------- 1 root root  71M Oct 14 08:46 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K Oct 14 08:46 chat_template.jinja
-rw-r--r-- 1 root root 5.1K Oct 14 08:46 README.md
-rw-r--r-- 1 root root  693 Oct 14 08:46 tokenizer_config.json
-rw-r--r-- 1 root root  11M Oct 14 08:46 tokenizer.json


In [24]:
!N_EVAL=200 python eval.py /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider finetuned outputs/adapter

^C
Traceback (most recent call last):
  File "/usr/lib/python3.13/pathlib/_local.py", line 231, in __str__
    return self._str
           ^^^^^^^^^
AttributeError: 'pathlib._local.PosixPath' object has no attribute '_str'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/kaggle/working/eval.py", line 31, in <module>
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
  File "/usr/local/lib/python3.13/dist-packages/transformers/__init__.py", line 30, in <module>
    from . import dependency_versions_check
  File "/usr/local/lib/python3.13/dist-packages/transformers/dependency_versions_check.py", line 16, in <module>
    from .utils.versions import require_version, require_version_core
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/__init__.py", line 22, in <module>
    from .auto_docstring import (
    ...<10 lines>...
    )
  File "/usr/local/lib/python3.13/dist-package

In [25]:
%cd /kaggle/working
!N_EVAL=200 python eval.py /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider finetuned_retry outputs/adapter

/kaggle/working
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 317.05it/s]
8/200  first-attempt acc: 0.875  with retry: 0.875
16/200  first-attempt acc: 0.750  with retry: 0.750
24/200  first-attempt acc: 0.792  with retry: 0.833
32/200  first-attempt acc: 0.812  with retry: 0.844
40/200  first-attempt acc: 0.750  with retry: 0.775
48/200  first-attempt acc: 0.708  with retry: 0.750
56/200  first-attempt acc: 0.679  with retry: 0.732
64/200  first-attempt acc: 0.641  with retry: 0.688
72/200  first-attempt acc: 0.583  with retry: 0.625
80/200  first-attempt acc: 0.562  with retry: 0.625
88/200  first-attempt acc: 0.568  with retry: 0.625
96/200  first-attempt acc: 0.562  with retry: 0.635
104/200  first-attempt acc: 0.519  with retry: 0.587
112/200  first-attempt acc: 0.491  with retry: 0.554
120/200  first-attempt acc: 0.475  with retry: 0.533
128/200  first-attempt acc: 0.477  with retry: 0.531
136/200  first-attempt acc: 0.456  with retry: 0.515
144/200  first-

In [26]:
import json

b = json.load(open("outputs/results_baseline.json"))["results"]
f = json.load(open("outputs/results_finetuned_retry.json"))["results"]

def kind(g):
    g = g.upper()
    if g.count("SELECT") > 1: return "nested/subquery"
    if "JOIN" in g: return "join"
    if "GROUP BY" in g: return "group by"
    if "ORDER BY" in g: return "order by"
    return "simple"

print("Baseline:  SQL errors =", sum(1 for x in b if x["error"]), "| wrong result =", sum(1 for x in b if not x["correct"] and not x["error"]))
print("Finetuned: SQL errors =", sum(1 for x in f if x["first_error"]), "| wrong result =", sum(1 for x in f if not x["first_correct"] and not x["first_error"]))

print("\nAccuracy by query type (correct/total):")
for name, rs, key in [("baseline ", b, "correct"), ("finetuned", f, "first_correct")]:
    tot, ok = {}, {}
    for x in rs:
        k = kind(x["gold"]); tot[k] = tot.get(k, 0) + 1; ok[k] = ok.get(k, 0) + x[key]
    print(name, {k: f"{ok[k]}/{tot[k]}" for k in sorted(tot)})

print("\n--- 5 fine-tuned failures ---")
fails = [x for x in f if not x["first_correct"]]
for x in fails[:5]:
    print("\nQ:", x["question"]); print("GOLD:", x["gold"]); print("PRED:", x["first_pred"]); print("ERR:", x["first_error"])

Baseline:  SQL errors = 64 | wrong result = 51
Finetuned: SQL errors = 57 | wrong result = 42

Accuracy by query type (correct/total):
baseline  {'group by': '5/16', 'join': '17/72', 'nested/subquery': '11/35', 'order by': '5/16', 'simple': '47/61'}
finetuned {'group by': '11/16', 'join': '15/72', 'nested/subquery': '11/35', 'order by': '9/16', 'simple': '55/61'}

--- 5 fine-tuned failures ---

Q: Show the name and the release year of the song by the youngest singer.
GOLD: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
PRED: SELECT T2.song_name , T2.song_release_year FROM singer AS T1 JOIN song AS T2 ON T1.singer_id = T2.singer_id ORDER BY T1.age LIMIT 1
ERR: no such table: song

Q: How many singers are from each country?
GOLD: SELECT country ,  count(*) FROM singer GROUP BY country
PRED: SELECT count(*) , country FROM singer GROUP BY country
ERR: None

Q: List all song names by singers above the average age.
GOLD: SELECT song_name FROM singer WHERE age  >  (SEL